In [0]:
-- ============================================================
-- CATALOG & SCHEMA SETUP
-- ============================================================
CREATE CATALOG IF NOT EXISTS workspace;

CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;

-- ============================================================
-- BRONZE LAYER — raw ingested tables
-- ============================================================

CREATE TABLE IF NOT EXISTS workspace.bronze.customers (
   customer_id BIGINT,
  first_name STRING COLLATE UTF8_BINARY,
  signup_date DATE,
  region STRING COLLATE UTF8_BINARY,
  customer_segment STRING COLLATE UTF8_BINARY,
  _ingested_at TIMESTAMP,
  _ingested_by STRING COLLATE UTF8_BINARY
) USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.bronze.products (
    product_id BIGINT,
  category STRING COLLATE UTF8_BINARY,
  unit_price DECIMAL(10,2),
  cost_price DECIMAL(10,2),
  _ingested_at TIMESTAMP,
  _ingested_by STRING COLLATE UTF8_BINARY
) USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.bronze.orders (
     order_id BIGINT,
  customer_id BIGINT,
  product_id BIGINT,
  quantity INT,
  order_date DATE,
  order_status STRING COLLATE UTF8_BINARY,
  payment_method STRING COLLATE UTF8_BINARY,
  _ingested_at TIMESTAMP,
  _source_file STRING COLLATE UTF8_BINARY
) USING DELTA;

-- ============================================================
-- SILVER LAYER — cleaned, joined, enriched
-- ============================================================

CREATE TABLE IF NOT EXISTS workspace.silver.sales_enriched (
    order_id                BIGINT,
    order_date              DATE,
    order_status            STRING,
    payment_method          STRING,
    quantity                INT,
    customer_id             BIGINT,
    region                  STRING,
    customer_segment        STRING,
    product_id              BIGINT,
    category                STRING,
    unit_price               DECIMAL(10,2),
    cost_price               DECIMAL(10,2),
    gross_amount             DECIMAL(12,2),
    cost_amount               DECIMAL(12,2),
    profit_amount             DECIMAL(12,2),
    is_revenue_recognized    BOOLEAN,
    order_year               INT,
    order_month              INT
) USING DELTA
PARTITIONED BY (order_year, order_month);

-- ============================================================
-- GOLD LAYER — aggregated reporting tables
-- ============================================================

CREATE TABLE IF NOT EXISTS workspace.gold.daily_sales_summary (
    order_date      DATE,
    category        STRING,
    total_orders    BIGINT,
    total_quantity  BIGINT,
    gross_revenue   DECIMAL(14,2),
    total_profit    DECIMAL(14,2)
) USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.gold.top_customers (
    customer_id       BIGINT,
    region            STRING,
    customer_segment  STRING,
    total_orders      BIGINT,
    lifetime_revenue  DECIMAL(14,2)
) USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.gold.monthly_category_performance (
    order_year      INT,
    order_month     INT,
    category        STRING,
    gross_revenue   DECIMAL(14,2),
    total_profit    DECIMAL(14,2),
    margin_pct      DECIMAL(5,2)
) USING DELTA;